[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-10-capstone-ml-schema-layer.ipynb#scrollTo=cc000001)

---
# Day 10 · Capstone — Type-Safe Config and Schema Layer for an ML Pipeline
**certified-journeys / pydantic-certified** · Exam · Full-Course Capstone

> **Goal for today:** Design and implement a production-grade Pydantic schema layer for an ML pipeline — covering BaseSettings, discriminated unions, cross-field validators, computed fields, serialization round-trips, JSON Schema export, and a FastAPI endpoint tested with TestClient.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings fastapi httpx


## Step 1 · Data and Training Config with BaseSettings

**`BaseSettings`** (from `pydantic-settings`) extends `BaseModel` with environment variable
loading. Fields are populated from:
1. Explicit keyword arguments (highest priority)
2. Environment variables (e.g. `DATA_ROOT_PATH`)
3. `.env` files
4. Field defaults (lowest priority)

In production, your ML pipeline reads its config from environment variables or
a secrets manager. In this notebook, we pass values explicitly.

**Docs:** https://docs.pydantic.dev/latest/concepts/pydantic_settings/


In [ ]:
from __future__ import annotations
import hashlib
import json
from typing import Annotated, List, Literal, Optional, Union
from pydantic import BaseModel, ConfigDict, Field, computed_field, field_validator, model_validator
from pydantic_settings import BaseSettings


class DataConfig(BaseSettings):
    """Data pipeline configuration — can be overridden by environment variables."""
    model_config = ConfigDict(env_prefix="DATA_")  # reads DATA_ROOT_PATH, DATA_TEST_SIZE, etc.

    root_path: str = Field(default="/data", description="Root path for datasets")
    train_file: str = Field(default="train.parquet")
    test_file: str  = Field(default="test.parquet")
    test_size: float = Field(default=0.2, gt=0, lt=1.0, description="Fraction of data held out for test")
    val_size: float  = Field(default=0.1, gt=0, lt=1.0, description="Fraction held out for validation")
    random_seed: int = Field(default=42)
    feature_columns: List[str] = Field(default_factory=list)
    target_column: str = Field(default="label")

    @model_validator(mode="after")
    def splits_must_not_exceed_one(self) -> DataConfig:
        if self.test_size + self.val_size >= 1.0:
            raise ValueError(
                f"test_size ({self.test_size}) + val_size ({self.val_size}) must be < 1.0 "
                f"to leave data for training."
            )
        return self


# Valid config
data_cfg = DataConfig(
    root_path="/datasets/project-x",
    feature_columns=["age", "income", "score"],
    test_size=0.2,
    val_size=0.1,
)
print(f"DataConfig:")
print(f"  root_path:       {data_cfg.root_path}")
print(f"  test_size:       {data_cfg.test_size}")
print(f"  val_size:        {data_cfg.val_size}")
print(f"  train fraction:  {1 - data_cfg.test_size - data_cfg.val_size:.0%}")
print(f"  feature_columns: {data_cfg.feature_columns}")

# Cross-field violation
from pydantic import ValidationError
try:
    DataConfig(test_size=0.7, val_size=0.5)
except ValidationError as e:
    print(f"\nSplit error: {e.errors()[0]['msg']}")


### What just happened?

- **`BaseSettings`** with `env_prefix="DATA_"` means `DATA_TEST_SIZE=0.3` in the environment would override the default — zero code changes between dev and production.
- **Cross-field `model_validator(mode='after')`** runs after all individual fields are validated — the right place for multi-field constraints.
- The validator raises a clear, actionable message: `test_size (0.7) + val_size (0.5) must be < 1.0`.
- In production, `DataConfig()` would read from `os.environ` or a `.env` file automatically.


## Step 2 · Discriminated Union `ModelSpec`

Different model types (linear, tree, neural) have completely different hyperparameters.
A discriminated union lets us express this cleanly: one field selects the variant,
and each variant has its own strongly-typed fields.


In [ ]:
class LinearModelSpec(BaseModel):
    model_type: Literal["linear"]
    regularization: Literal["l1", "l2", "elasticnet"] = "l2"
    C: float = Field(default=1.0, gt=0, description="Inverse regularization strength")
    max_iter: int = Field(default=1000, ge=10)
    fit_intercept: bool = True


class TreeModelSpec(BaseModel):
    model_type: Literal["tree"]
    n_estimators: int = Field(default=100, ge=1)
    max_depth: Optional[int] = Field(default=None, ge=1, le=100)
    min_samples_split: int = Field(default=2, ge=2)
    learning_rate: Optional[float] = Field(default=None, gt=0, lt=1)
    boosting: bool = False


class NeuralModelSpec(BaseModel):
    model_type: Literal["neural"]
    hidden_layers: List[int] = Field(default=[128, 64], min_length=1)
    activation: Literal["relu", "tanh", "sigmoid"] = "relu"
    dropout: float = Field(default=0.0, ge=0.0, le=0.9)
    batch_norm: bool = False


# Discriminated union — O(1) dispatch on model_type
ModelSpec = Annotated[
    Union[LinearModelSpec, TreeModelSpec, NeuralModelSpec],
    Field(discriminator="model_type"),
]

from pydantic import TypeAdapter
model_spec_adapter = TypeAdapter(ModelSpec)

# Test each variant
for raw in [
    {"model_type": "linear", "regularization": "l1", "C": 0.5},
    {"model_type": "tree",   "n_estimators": 200, "max_depth": 6, "boosting": True},
    {"model_type": "neural", "hidden_layers": [256, 128, 64], "dropout": 0.3},
]:
    spec = model_spec_adapter.validate_python(raw)
    print(f"{type(spec).__name__:20s}: {spec.model_dump()}")


### What just happened?

- Each model variant has **fully typed, validated hyperparameters** — `max_depth` only exists on `TreeModelSpec`, `hidden_layers` only on `NeuralModelSpec`.
- The discriminated union **routes by `model_type` in O(1)** — no trial-and-error across all three schemas.
- `TypeAdapter(ModelSpec)` lets us validate a bare `ModelSpec` dict without a wrapper class — clean for reuse.
- Adding a new model type is one step: define the class and add it to the `Union` — no branching logic anywhere else.


## Step 3 · Training and Inference Config

The training config holds optimizer settings and cross-field constraints;
the inference config holds serving parameters.


In [ ]:
class TrainingConfig(BaseModel):
    epochs: int = Field(default=50, ge=1)
    batch_size: int = Field(default=32, ge=1)
    optimizer: Literal["adam", "sgd", "rmsprop"] = "adam"
    learning_rate: float = Field(default=1e-3, gt=0)
    early_stopping_patience: Optional[int] = Field(default=None, ge=1)
    checkpoint_every_n_epochs: int = Field(default=10, ge=1)

    @model_validator(mode="after")
    def adam_requires_sensible_lr(self) -> TrainingConfig:
        """Adam performs poorly with very large learning rates."""
        if self.optimizer == "adam" and self.learning_rate > 0.1:
            raise ValueError(
                f"When optimizer='adam', learning_rate must be <= 0.1 "
                f"(got {self.learning_rate}). Consider lr=1e-3."
            )
        return self


class InferenceConfig(BaseModel):
    batch_size: int = Field(default=128, ge=1)
    max_latency_ms: Optional[float] = Field(default=None, gt=0)
    confidence_threshold: float = Field(default=0.5, ge=0.0, le=1.0)
    return_probabilities: bool = True
    device: Literal["cpu", "cuda", "mps"] = "cpu"


# Valid training config
train_cfg = TrainingConfig(epochs=100, optimizer="adam", learning_rate=1e-3)
print(f"TrainingConfig: optimizer={train_cfg.optimizer}, lr={train_cfg.learning_rate}")

# Cross-field violation: adam + high lr
try:
    TrainingConfig(optimizer="adam", learning_rate=0.5)
except ValidationError as e:
    print(f"\nLR violation: {e.errors()[0]['msg']}")

# SGD + same high lr is fine
sgd_cfg = TrainingConfig(optimizer="sgd", learning_rate=0.5)
print(f"SGD high LR is fine: {sgd_cfg.learning_rate}")

infer_cfg = InferenceConfig(batch_size=256, confidence_threshold=0.7, device="cpu")
print(f"\nInferenceConfig: {infer_cfg.model_dump()}")


### What just happened?

- **Optimizer-conditional validation**: `adam_requires_sensible_lr` only fires when `optimizer == 'adam'` — it's a business rule, not a data type constraint.
- The same learning rate (0.5) is accepted for SGD but rejected for Adam — context-dependent validation.
- `InferenceConfig` separates inference-time concerns from training — clean separation of concerns across the pipeline stages.
- Both configs are independently validated and composable into larger structures.


## Step 4 · `PipelineRun` with `computed_field` for Deterministic Run ID

**`computed_field`** adds a read-only property to the model's serialized output.
We use it to generate a deterministic `run_id` by hashing the serialized config —
the same config always produces the same `run_id`, enabling cache-friendly reruns.


In [ ]:
class PipelineRun(BaseModel):
    """Top-level ML pipeline run configuration."""
    name: str = Field(description="Human-readable run name")
    data: DataConfig
    model_spec: ModelSpec
    training: TrainingConfig
    inference: InferenceConfig
    tags: List[str] = Field(default_factory=list)
    notes: Optional[str] = None

    @computed_field
    @property
    def run_id(self) -> str:
        """Deterministic hash of the config — same config → same run_id."""
        # Exclude non-deterministic or human-readable fields from the hash
        payload = self.model_dump(
            exclude={"name", "tags", "notes", "run_id"},
            mode="json",
        )
        serialized = json.dumps(payload, sort_keys=True)
        return "run-" + hashlib.sha256(serialized.encode()).hexdigest()[:12]


PipelineRun.model_rebuild()  # resolve forward references in nested models

run = PipelineRun(
    name="experiment-001",
    data=DataConfig(
        root_path="/data/project-x",
        feature_columns=["age", "income", "score"],
    ),
    model_spec={"model_type": "tree", "n_estimators": 200, "max_depth": 8},
    training=TrainingConfig(epochs=50, optimizer="sgd", learning_rate=0.01),
    inference=InferenceConfig(confidence_threshold=0.6),
    tags=["baseline", "tree"],
)

print(f"Run name:  {run.name}")
print(f"Run ID:    {run.run_id}")
print(f"Model:     {run.model_spec.model_type}")
print(f"Estimators:{run.model_spec.n_estimators}")

# Same config → same run_id (deterministic)
run2 = PipelineRun(
    name="experiment-002-renamed",   # name is excluded from hash
    data=DataConfig(root_path="/data/project-x", feature_columns=["age", "income", "score"]),
    model_spec={"model_type": "tree", "n_estimators": 200, "max_depth": 8},
    training=TrainingConfig(epochs=50, optimizer="sgd", learning_rate=0.01),
    inference=InferenceConfig(confidence_threshold=0.6),
)
print(f"\nrun.run_id  == run2.run_id: {run.run_id == run2.run_id}")
print(f"(Different name, same model config → same hash)")


### What just happened?

- **`computed_field`** makes `run_id` appear in `model_dump()` and `model_dump_json()` output — it's part of the schema, not a hidden method.
- **Deterministic hashing**: excluding human-readable fields (`name`, `tags`) from the hash means the run ID tracks the *config*, not arbitrary metadata.
- Two runs with different names but identical configs produce the same `run_id` — enabling cache-hit detection and experiment deduplication.
- `model_dump(mode='json')` ensures all values are JSON-serializable before hashing.


## Step 5 · Serialization Round-Trip

A critical test for any schema layer: serialize → deserialize → assert equal.
This confirms that `model_dump_json()` and `model_validate_json()` are inverse operations.


In [ ]:
# Serialize to JSON
run_json = run.model_dump_json(indent=2)
print(f"Serialized ({len(run_json)} chars, first 300):")
print(run_json[:300])
print("...")

# Deserialize from JSON
run_restored = PipelineRun.model_validate_json(run_json)

# Assert round-trip equality
assert run_restored.run_id == run.run_id, "run_id mismatch!"
assert run_restored.model_spec.model_type == run.model_spec.model_type
assert run_restored.training.epochs == run.training.epochs
assert run_restored.data.feature_columns == run.data.feature_columns

# Deep equality via model_dump
original_dump   = run.model_dump(mode="json")
restored_dump   = run_restored.model_dump(mode="json")
assert original_dump == restored_dump, "Dump mismatch!"

print("\nRound-trip assertions all passed.")
print(f"run_id preserved: {run_restored.run_id}")
print(f"Model type preserved: {run_restored.model_spec.model_type}")
print(f"Feature columns: {run_restored.data.feature_columns}")


### What just happened?

- **`model_dump_json()`** produces a JSON string with all nested models, computed fields, and discriminated union variants serialized correctly.
- **`model_validate_json()`** reconstructs the full `PipelineRun` including the correct discriminated union variant (`TreeModelSpec`) from the `model_type` field.
- The `run_id` computed field is recomputed on restore — and matches, proving the config is identical.
- Round-trip testing is essential before putting a schema layer into production.


## Step 6 · JSON Schema Export

Export the full JSON Schema for `PipelineRun`. This schema can be:
- Shared with frontend teams to validate config forms
- Used in CI to lint experiment configs before they run
- Registered with a schema registry (e.g. Confluent Schema Registry, AWS Glue)
- Served from a `/schema` endpoint


In [ ]:
import json

schema = PipelineRun.model_json_schema()

# Show top-level structure
print("Top-level schema keys:")
print(json.dumps({k: v for k, v in schema.items() if k != "$defs"}, indent=2))

# Show component definitions (the $defs section)
print(f"\nComponent definitions ($defs): {sorted(schema.get('$defs', {}).keys())}")

# Check the discriminated union appears correctly
model_spec_schema = schema["$defs"].get("LinearModelSpec") or schema["properties"].get("model_spec")
print(f"\nLinearModelSpec in schema: {'LinearModelSpec' in schema.get('$defs', {})}")
print(f"TreeModelSpec in schema:   {'TreeModelSpec' in schema.get('$defs', {})}")
print(f"NeuralModelSpec in schema: {'NeuralModelSpec' in schema.get('$defs', {})}")

# Save to file (useful for CI/CD pipeline integration)
schema_path = "/tmp/pipeline_run_schema.json"
with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)

print(f"\nSchema saved to {schema_path}")
print(f"Total schema size: {len(json.dumps(schema)):,} chars")


### What just happened?

- **`model_json_schema()`** generates a complete JSON Schema including `$defs` for all nested models — one call, full schema.
- All three `ModelSpec` variants appear as separate `$defs` entries, linked from the discriminated union's `oneOf` or `anyOf`.
- The schema captures all constraints: `gt`, `ge`, `lt`, `le`, `min_length`, `pattern`, `Literal` enums — everything.
- Saving to a file enables version-controlled schema evolution and CI-based schema diffing.


## Step 7 · FastAPI `/run` Endpoint with TestClient

Expose the schema layer through a FastAPI endpoint.
The endpoint accepts a `PipelineRun` body, validates it, and returns a structured launch response.
We also expose `/schema` to serve the JSON Schema.


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel as PydanticBaseModel


class LaunchResponse(PydanticBaseModel):
    run_id: str
    name: str
    model_type: str
    status: Literal["queued", "running", "failed"] = "queued"
    message: str


ml_app = FastAPI(
    title="ML Pipeline API",
    description="Type-safe ML pipeline runner powered by Pydantic",
    version="1.0.0",
)

# In-memory store of launched runs
_launched_runs: dict[str, dict] = {}


@ml_app.post("/run", response_model=LaunchResponse, status_code=202)
def launch_run(payload: PipelineRun) -> LaunchResponse:
    """Accept a fully-validated pipeline config and queue the run."""
    run_id = payload.run_id

    if run_id in _launched_runs:
        # Idempotent: same config → same run_id → already queued
        raise HTTPException(
            status_code=409,
            detail={"error": "already_exists", "run_id": run_id, "name": payload.name},
        )

    _launched_runs[run_id] = payload.model_dump(mode="json")

    return LaunchResponse(
        run_id=run_id,
        name=payload.name,
        model_type=payload.model_spec.model_type,
        status="queued",
        message=f"Run '{payload.name}' queued with {payload.training.epochs} epochs.",
    )


@ml_app.get("/run/{run_id}")
def get_run(run_id: str) -> dict:
    if run_id not in _launched_runs:
        raise HTTPException(404, detail={"error": "not_found", "run_id": run_id})
    return _launched_runs[run_id]


@ml_app.get("/schema")
def get_schema() -> dict:
    """Serve the JSON Schema for PipelineRun."""
    return PipelineRun.model_json_schema()


ml_client = TestClient(ml_app)

# ── Test 1: Launch a valid run ────────────────────────────────────────────────
run_payload = {
    "name": "experiment-neural-001",
    "data": {
        "root_path": "/data/mnist",
        "feature_columns": ["pixel_mean", "pixel_std"],
        "test_size": 0.2,
        "val_size": 0.1,
    },
    "model_spec": {
        "model_type": "neural",
        "hidden_layers": [256, 128],
        "dropout": 0.2,
    },
    "training": {
        "epochs": 20,
        "optimizer": "adam",
        "learning_rate": 0.001,
        "batch_size": 64,
    },
    "inference": {"confidence_threshold": 0.7, "device": "cpu"},
    "tags": ["neural", "mnist"],
}

r1 = ml_client.post("/run", json=run_payload)
print(f"Launch: {r1.status_code}")
launch_body = r1.json()
print(f"Response: {launch_body}")
launched_run_id = launch_body["run_id"]

# ── Test 2: Duplicate run (same config → same run_id → 409) ──────────────────
r2 = ml_client.post("/run", json=run_payload)
print(f"\nDuplicate: {r2.status_code} → {r2.json()}")

# ── Test 3: Get the stored run ────────────────────────────────────────────────
r3 = ml_client.get(f"/run/{launched_run_id}")
stored = r3.json()
print(f"\nStored run model_type: {stored['model_spec']['model_type']}")
print(f"Stored run epochs:     {stored['training']['epochs']}")

# ── Test 4: Validation error (bad learning rate with adam) ───────────────────
bad_payload = dict(run_payload)
bad_payload["training"] = {"optimizer": "adam", "learning_rate": 0.5, "epochs": 10}
r4 = ml_client.post("/run", json=bad_payload)
print(f"\nValidation error: {r4.status_code}")
for err in r4.json().get("detail", []):
    print(f"  [{' → '.join(str(p) for p in err['loc'])}] {err['msg']}")


### What just happened?

- **`PipelineRun` as the request body**: FastAPI validates the full nested structure — data splits, model hyperparameters, training config — in one call.
- **Idempotent launch**: duplicate configs produce the same `run_id`, so the 409 check prevents accidental re-queuing.
- **422 from cross-field validator**: the Adam + high LR constraint fires during request parsing — the endpoint never runs.
- **`/schema` endpoint** serves the live JSON Schema — a self-documenting API contract.


## Step 8 · Pydantic Best Practices Review

A final look at the patterns used in this capstone and when to apply each.

**Docs:** https://docs.pydantic.dev/latest/concepts/best_practices/


In [ ]:
# Summary: verify the full pipeline works end-to-end
import json

print("=" * 60)
print("Pydantic ML Schema Layer — End-to-End Verification")
print("=" * 60)

# 1. Build a PipelineRun with all three model spec variants
for model_cfg in [
    {"model_type": "linear", "regularization": "l2", "C": 1.0},
    {"model_type": "tree",   "n_estimators": 100, "max_depth": 5},
    {"model_type": "neural", "hidden_layers": [128, 64], "dropout": 0.1},
]:
    pr = PipelineRun(
        name=f"run-{model_cfg['model_type']}",
        data=DataConfig(feature_columns=["a", "b", "c"]),
        model_spec=model_cfg,
        training=TrainingConfig(epochs=10, optimizer="sgd", learning_rate=0.01),
        inference=InferenceConfig(),
    )
    # Round-trip
    restored = PipelineRun.model_validate_json(pr.model_dump_json())
    assert restored.run_id == pr.run_id
    assert restored.model_spec.model_type == pr.model_spec.model_type
    print(f"  {model_cfg['model_type']:8s}: run_id={pr.run_id}, round-trip OK")

# 2. Verify JSON Schema contains all expected components
schema = PipelineRun.model_json_schema()
defs = set(schema.get("$defs", {}).keys())
required_defs = {"DataConfig", "TrainingConfig", "InferenceConfig",
                 "LinearModelSpec", "TreeModelSpec", "NeuralModelSpec"}
missing = required_defs - defs
print(f"\nJSON Schema $defs present: {not missing}")
if missing:
    print(f"  Missing: {missing}")

# 3. TestClient /schema endpoint
schema_resp = ml_client.get("/schema")
assert schema_resp.status_code == 200
print(f"  /schema endpoint: {schema_resp.status_code} OK")

print("\nAll checks passed.")


### What just happened?

- All three `ModelSpec` variants serialized, round-tripped, and validated correctly.
- The JSON Schema `$defs` section contains every component — proving the schema is complete and self-referential.
- The `/schema` endpoint returned 200 with the live schema.
- This end-to-end test is the kind of verification you'd run in CI before deploying a new pipeline config version.


In [ ]:
# Challenge: Extend the ML schema layer
#
# Requirements:
#
#   1. Add a new ModelSpec variant: EnsembleModelSpec
#        model_type: Literal['ensemble']
#        members: List[ModelSpec]   ← nested discriminated union!
#        voting: Literal['hard', 'soft'] = 'soft'
#      Call model_rebuild() on EnsembleModelSpec after defining it.
#
#   2. Add a PreprocessingConfig model:
#        normalize: bool = True
#        imputation_strategy: Literal['mean', 'median', 'drop'] = 'mean'
#        max_missing_fraction: float = Field(default=0.3, ge=0, le=1)
#      Add a validator: if imputation_strategy == 'drop', warn (don't error) that
#        rows with missing values will be removed — you can just print a warning.
#
#   3. Add PreprocessingConfig as a field to PipelineRun (with a default).
#
#   4. Build a PipelineRun with an EnsembleModelSpec containing two members:
#        a LinearModelSpec and a TreeModelSpec.
#        Verify the round-trip and print the run_id.
#
#   5. Post it to /run with TestClient and verify 202 Accepted.
#

# Your solution here
# class EnsembleModelSpec(BaseModel): ...
# EnsembleModelSpec.model_rebuild()
# ... update ModelSpec union ...
# class PreprocessingConfig(BaseModel): ...
# PipelineRun — add preprocessing field ...
# PipelineRun.model_rebuild()
# ensemble_run = PipelineRun(...)
# ...


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `BaseSettings` | Loads from env vars and `.env`; `env_prefix` namespaces the vars |
| Discriminated union `ModelSpec` | O(1) dispatch; each variant has its own typed fields |
| Cross-field `model_validator` | `mode='after'` for multi-field rules; `mode='before'` for pre-parse transforms |
| `computed_field` | Read-only property that appears in `model_dump()` and JSON output |
| Deterministic `run_id` | Hash config (excluding metadata) → same config → same ID |
| Round-trip assertion | `model_dump_json()` → `model_validate_json()` → assert equal |
| `model_json_schema()` | Single source of truth → OpenAPI + JSON Schema + config linting |
| TestClient in notebook | No uvicorn needed; full integration test in-process |

> **Tip:** A Pydantic schema layer is the single source of truth for your ML pipeline: the same models drive runtime validation, JSON Schema, OpenAPI docs, and `.env` configuration — four concerns, one definition.

---
## Course complete!

You've covered the full Pydantic v2 stack:

- **Days 1–3**: Core models, field validation, custom validators
- **Days 4–6**: Settings, serialization, advanced validators
- **Days 7–9**: Nested models, discriminated unions, recursive types, FastAPI integration, performance patterns
- **Day 10**: Production-grade ML schema layer combining everything

Mark Day 10 complete in your [tracker](../index.html).
